# STAC AI Workshop · NYC · 21 October 2026
## 02 · Building grounded financial RAG — guided walkthrough
**Matthew Dixon, PhD, FRM · Beginner workbook · revised 21 October 2026**

**Our question:** Can Harbor Metals describe its covenant headroom as comfortable?

Run each code cell with **Shift+Enter**, from top to bottom. Read the instruction above it and the interpretation below it. All default steps work without a key or network. Documents are fictional. Keep `section2_rag_support.py` with this notebook.

You will: inspect documents → send a search query → inspect returned passages → filter sources → inspect the model's exact input → trace and repair missing support → measure retrieval quality.

**Why learn this if we have LLMs?** They reduce the need to build separate NLP models, but they can still confuse total cash with eligible cash or forecast earnings with reported earnings. We need to understand what text reaches them.

**RAG means retrieval-augmented generation:** search returns text; that text is put into a model prompt; the model produces an answer. Our default executes retrieval and evidence checks locally. The optional final cell performs real generation using your provider key; the default displays a clearly labelled worked answer.

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/02_grounded_rag.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/02_grounded_rag.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

## 0 · Set up once

Run this cell. It imports display and search helpers so the rest of the notebook stays short.

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:7** | `DOCUMENTS, table, search_documents, show_results, filter_documents,` | Execute search and return passages; inspect IDs and text, not only scores. |
| **C01:8** | `assemble_context, check_requirements, show_graph, evaluate_queries,` | Build the exact question-plus-evidence input supplied to the generator. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys
folder = next(p for p in [Path.cwd(), Path.cwd()/"notebooks"] if (p/"section2_rag_support.py").exists())
sys.path.insert(0, str(folder))
from section2_rag_support import (
    DOCUMENTS, table, search_documents, show_results, filter_documents,
    assemble_context, check_requirements, show_graph, evaluate_queries,
)
print("Ready. No API calls have been made.")

**What to notice:** “Ready” means setup succeeded. You do not need to inspect the helper file to follow the exercise.

## 1 · Which documents can search access?

`DOCUMENTS` is the complete searchable collection. Search cannot retrieve anything outside this list. Read the title, issuer and availability columns.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:2** | `table([{"ID": d["id"], "Title": d["title"], "Issuer": d["issuer"], "Available (New York)": d["available"]} for d in DOCUMENTS])` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
table([{"ID": d["id"], "Title": d["title"], "Issuer": d["issuer"], "Available (New York)": d["available"]} for d in DOCUMENTS])

**What to notice:** There is a different issuer (`OTHER`) and a later amendment (`AMENDMENT`). A matching phrase does not make either appropriate for our decision.

## 2 · Read the actual source text

Change `document_id` to `FILING`, `COVENANT` or `ANNEX`. Rerun this cell to read that document.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `document_id = "COVENANT"  # TRY: "FILING" or "ANNEX"` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
document_id = "COVENANT"  # TRY: "FILING" or "ANNEX"
document = next(d for d in DOCUMENTS if d["id"] == document_id)
print(document["title"]); print(); print(document["text"])

**What to notice:** The covenant defines eligible cash and refers to Annex A. This is why the filing alone cannot determine the covenant ratio. Our passages are kept whole; splitting off the exception would lose meaning.

## 3 · Send a query to the retrieval tool

The function name is **`search_documents`**. It takes a query, the documents it may search and a maximum result count. It returns ranked passages, not an answer. Edit the query and rerun.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:3** | `results = search_documents(query=query, documents=DOCUMENTS, top_k=4)` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
query = "restricted cash covenant"  # TRY: "cash unavailable for debt repayment"
results = search_documents(query=query, documents=DOCUMENTS, top_k=4)
print("QUERY SENT TO SEARCH:", query)
show_results(results)

**What to notice:** Read **ID**, **matched words** and **returned passage**. The score is lexical TF-IDF cosine similarity, not a probability of truth. The later amendment or wrong issuer can rank highly. We have not called an LLM.

## 4 · Limit evidence to the right issuer and time

Our decision is at **08:55 New York**. This cell applies explicit metadata filters before retrieval. Change the cutoff to `2026-10-20 09:10` to see the amendment become available, then restore 08:55.

<!-- WORKSHOP-CODE-GUIDE C05 -->
### Code cell C05 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C05:3** | `eligible_documents, exclusions = filter_documents(DOCUMENTS, issuer="Harbor Metals", cutoff=cutoff)` | Apply the explicit issuer and availability rules before using evidence. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C05 · code cell
cutoff = "2026-10-20 08:55"
eligible_documents, exclusions = filter_documents(DOCUMENTS, issuer="Harbor Metals", cutoff=cutoff)
table(exclusions)

**What to notice:** `OTHER` is excluded for issuer and `AMENDMENT` for availability. Availability is different from the financial period a document describes. These are declared filters, not an LLM judgment. All dates in this small exercise use the same timezone and sortable format.

## 5 · Rerun the same search after filtering

Use the same query and compare IDs with Step 3.

<!-- WORKSHOP-CODE-GUIDE C06 -->
### Code cell C06 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C06:2** | `eligible_results = search_documents(query=query, documents=eligible_documents, top_k=4)` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C06 · code cell
eligible_results = search_documents(query=query, documents=eligible_documents, top_k=4)
show_results(eligible_results)

**What to notice:** The search now operates only on eligible documents. It may still miss a necessary passage: filtering improves eligibility, not completeness.

## 6 · Where dense retrieval fits
Our working search uses **words and TF-IDF**. Dense retrieval instead encodes the query and each passage into learned vectors, then ranks their similarity. “Cash unavailable for debt repayment” may match “restricted cash” even with different wording. Neither score establishes contractual applicability.

This follows the retriever–reader teaching structure in [Stanford CS224N, Question Answering](https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1254/slides_w25/cs224n-2025-lecture13-QA.pdf) and the two-encoder approach in [Dense Passage Retrieval](https://aclanthology.org/2020.emnlp-main.550/).

**Optional actual dense search:** the next cell is disabled by default. With `sentence-transformers` installed, enabling it downloads an embedding model on the first run and computes actual vectors locally. It does not use fabricated or relabelled lexical scores. The retrieval lesson and all later steps work without this optional download.

<!-- WORKSHOP-CODE-GUIDE C07 -->
### Code cell C07 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C07:2** | `RUN_DENSE = False  # Optional: install sentence-transformers, then change to True.` | This switch controls an optional live, model-download or training path. Leave it False for the default exercise. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C07 · code cell
RUN_DENSE = False  # Optional: install sentence-transformers, then change to True.
if RUN_DENSE:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    passage_vectors = encoder.encode([d["text"] for d in eligible_documents], normalize_embeddings=True)
    query_vector = encoder.encode(query, normalize_embeddings=True)
    scores = passage_vectors @ query_vector
    ranked = sorted(zip(eligible_documents, scores), key=lambda pair: -pair[1])[:4]
    table([{"ID": d["id"], "Dense similarity": round(float(s), 3), "Passage": d["text"]} for d, s in ranked])
else:
    print("Optional dense search skipped. Continue to Step 7.")

**What to notice:** If enabled, compare document IDs with Step 5. A different ranking is not automatically a better ranking; later we compare results with declared relevant documents. This small general embedding model is not the original DPR model.

## 7 · Retrieve the pieces required for the question

A single vague query can miss a required clause. Here the application explicitly sends **three queries**. Read each query and the returned IDs. These steps are fixed in Python.

<!-- WORKSHOP-CODE-GUIDE C08 -->
### Code cell C08 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C08:5** | `passages = search_documents(search_query, eligible_documents, top_k=2)` | Execute search and return passages; inspect IDs and text, not only scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C08 · code cell
queries = ["reports debt cash EBITDA", "signed covenant eligible cash", "Annex A restricted cash"]
selected = {}
for search_query in queries:
    passages = search_documents(search_query, eligible_documents, top_k=2)
    print(search_query, "→", ", ".join(p["id"] for p in passages))
    for passage in passages:
        selected[passage["id"]] = passage
context_documents = list(selected.values())

**What to notice:** The application wrote the queries and fixed `top_k=2`. An agent could choose queries and repeat searches, but we would still inspect the returned sources. Multiple retrievals are not proof that everything needed was found.

## 8 · Inspect the exact model input

`assemble_context` places retrieved passages below the question. Read the output: this is the actual input we can send to the model, not hidden JSON.

<!-- WORKSHOP-CODE-GUIDE C09 -->
### Code cell C09 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C09:3** | `model_input = assemble_context(question, context_documents)` | Build the exact question-plus-evidence input supplied to the generator. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C09 · code cell
question = "What is Harbor Metals' current covenant leverage, and can we call its headroom comfortable?"
model_input = assemble_context(question, context_documents)
print(model_input)

**What to notice:** Look for `[FILING]`, `[COVENANT]` and `[ANNEX]`. The model sees the text under these IDs. It does not automatically see the rest of the document store. Source IDs help attribution but do not guarantee correct reasoning.

## 9 · Check declared evidence requirements

For this teaching workflow, a credit expert requires a filing, controlling covenant and annex before checking the current ratio; a comfort claim also requires downside evidence. These requirements are explicitly specified in the helper. They are not inferred by search.

<!-- WORKSHOP-CODE-GUIDE C10 -->
### Code cell C10 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C10:2** | `table(check_requirements(context_documents))` | Check the declared evidence prerequisites; presence is not certification of truth. |
| **C10:3** | `show_graph(context_documents)` | Display the recorded dependencies and any missing evidence path. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C10 · code cell
table(check_requirements(context_documents))
show_graph(context_documents)

**What to notice:** “READY TO CHECK” means required document IDs are present; it does not certify their contents. The comfort claim remains blocked because no downside assessment was supplied. Graph edges record dependencies that allow us to locate missing support.

## 10 · Bind numbers and call the calculator

Read the source values in Step 2. Here we bind the reviewed numbers explicitly; there is no hidden model extraction. The function name is **`calculate_leverage`**. Change restricted cash from 5 to 0 to see the effect, then restore 5.

<!-- WORKSHOP-CODE-GUIDE C11 -->
### Code cell C11 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C11:2** | `def calculate_leverage(debt, eligible_cash, ebitda):` | Execute the calculator with these arguments; inspect the cash definition. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C11 · code cell
def calculate_leverage(debt, eligible_cash, ebitda):
    return (debt - eligible_cash) / ebitda

debt = 75          # FILING, $ millions
cash = 15          # FILING
restricted_cash = 5  # ANNEX; COVENANT says to exclude it
eligible_cash = cash - restricted_cash
ratio = calculate_leverage(debt=debt, eligible_cash=eligible_cash, ebitda=20)
print(f"Calculator input: debt={debt}, eligible cash={eligible_cash}, EBITDA=20")
print(f"Calculator output: {ratio:.2f}×")
print(f"Distance below the 3.5× ceiling: {3.5-ratio:.2f}×")

**What to notice:** The expected ratio is **3.25×**, not 3.00×. Correct arithmetic requires the correct cash definition. Being 0.25× below the ceiling alone does not establish comfortable downside headroom.

## 11 · Remove the annex and trace the gap

This changes the evidence packet, not the numbers in the previous cell. Inspect the graph and requirement table. We must not reuse the old ratio as newly verified evidence.

<!-- WORKSHOP-CODE-GUIDE C12 -->
### Code cell C12 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C12:3** | `table(check_requirements(incomplete))` | Check the declared evidence prerequisites; presence is not certification of truth. |
| **C12:4** | `show_graph(incomplete)` | Display the recorded dependencies and any missing evidence path. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C12 · code cell
incomplete = [d for d in context_documents if d["id"] != "ANNEX"]
table(check_requirements(incomplete))
show_graph(incomplete)
print("Do not release a newly verified ratio: the restricted-cash input is unresolved.")

**What to notice:** The missing annex blocks the supported calculation path and ratio claim. The graph explains why; it does not guess the missing amount. This is a controlled dependency check, not automatic discovery of every possible evidence gap.

## 12 · Repair the gap

Retrieve the annex again from the eligible collection, then attach the actual returned source. A document from another issuer cannot pass our source filter.

<!-- WORKSHOP-CODE-GUIDE C13 -->
### Code cell C13 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C13:2** | `repair_results = search_documents("Annex A restricted cash", eligible_documents, top_k=3)` | Execute search and return passages; inspect IDs and text, not only scores. |
| **C13:6** | `table(check_requirements(repaired))` | Check the declared evidence prerequisites; presence is not certification of truth. |
| **C13:7** | `show_graph(repaired)` | Display the recorded dependencies and any missing evidence path. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C13 · code cell
repair_results = search_documents("Annex A restricted cash", eligible_documents, top_k=3)
repair = next(d for d in repair_results if d["id"] == "ANNEX")
print("REPAIR EVIDENCE:", repair["text"])
repaired = incomplete + [repair]
table(check_requirements(repaired))
show_graph(repaired)

**What to notice:** Current-ratio evidence becomes checkable again, but the comfort claim still needs downside evidence. Restoring one source repairs only the claims depending on it.

## 13 · Measure retrieval quality

These relevant-document labels were supplied for the teaching exercise. Edit a query and rerun. Precision@3 asks how many of three retrieval slots contain relevant documents; recall@3 asks how many required relevant documents were found.

<!-- WORKSHOP-CODE-GUIDE C14 -->
### Code cell C14 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C14:7** | `table(evaluate_queries(labelled_queries, eligible_documents, k=3))` | Compare retrieved IDs with the supplied relevance labels. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C14 · code cell
labelled_queries = [
    ("restricted cash unavailable repayment", {"COVENANT", "ANNEX"}),
    ("debt EBITDA quarterly filing", {"FILING"}),
    ("net leverage definition", {"COVENANT", "ANNEX"}),
]
table(evaluate_queries(labelled_queries, eligible_documents, k=3))

**What to notice:** Inspect **Missed**, not just the score. These labels and three queries are an instructional evaluation, not an independent benchmark. Production evaluation needs expert-labelled queries and unseen cases. See [Stanford IR: precision and recall](https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-of-unranked-retrieval-sets-1.html).

## 14 · Optional generation using OpenAI or Together

Default: inspect a worked answer without a key. To call a real model, set `provider` to `openai` or `together` and enter a model ID available to your account. A hidden prompt requests your key. This sends the fictional evidence packet to that provider.

<!-- WORKSHOP-CODE-GUIDE C15 -->
### Code cell C15 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C15:2** | `provider = "offline"  # "openai" or "together" to make a live call` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C15 · code cell
provider = "offline"  # "openai" or "together" to make a live call
model_name = ""        # Enter a model ID available to your provider account.
if provider == "offline":
    print("WORKED ANSWER — not a recorded model response")
    print("The reviewed covenant calculation is (75 − (15 − 5)) / 20 = 3.25×.")
    print("It is below the 3.5× ceiling. Comfortable headroom remains unsupported without downside evidence.")
else:
    from getpass import getpass
    from architecture_support import HostedLLM
    if not model_name:
        raise ValueError("Enter your provider's model ID in model_name first.")
    model = HostedLLM(provider, model_name, getpass("Your API key (hidden): "))
    response = model.complete([{"role": "user", "content": model_input}])
    print(response["content"])

**What to notice:** For a live answer, check whether it cites the annex, uses eligible cash of 10, calculates 3.25× and qualifies the comfort claim. A fluent answer can still fail any of these checks. Default execution makes no live call.

## Check your understanding
1. A passage scores highly but arrived after the cutoff. Can it support the historical assessment?
2. The ratio calculation is numerically correct but Annex A is missing. What should you do?
3. Why can a repaired ratio path still leave the comfort claim unsupported?

<details><summary>Reveal answers</summary>
1. No: search relevance does not override availability.<br>
2. Obtain and verify the controlling definition and annex, bind the eligible-cash input, then recompute.<br>
3. They require different evidence. Downside headroom needs scenario information in addition to the current calculation.
</details>

**Takeaway:** Inspect the query, returned text, eligibility and claim requirements. RAG supplies context; an evidence graph helps trace whether declared support requirements are met and what a repair changes.

### External teaching material used
- [Stanford CS224N: Question Answering](https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1254/slides_w25/cs224n-2025-lecture13-QA.pdf): retriever–reader structure.
- [Karpukhin et al., Dense Passage Retrieval](https://aclanthology.org/2020.emnlp-main.550/): separate query and passage embeddings.
- [Microsoft Azure AI Search: hybrid search](https://learn.microsoft.com/en-us/azure/search/hybrid-search-overview): combining lexical and vector candidates in a practical service.
- [Stanford Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/html/htmledition/evaluation-of-unranked-retrieval-sets-1.html): precision and recall.
- [Liu et al., Lost in the Middle](https://aclanthology.org/2024.tacl-1.9/): experimental evidence that context position can affect use of retrieved information. This notebook does not reproduce that experiment.

The financial examples and dependency checks are workshop adaptations, not results reported in those papers.

## Answer checks · default configuration

- Search is **lexical TF-IDF cosine**, not semantic reasoning or a calibrated confidence model. Query words and the chosen document scope determine candidates.
- The default cutoff is **08:55 New York**: `OTHER` fails issuer scope; the later `AMENDMENT` fails availability.
- Current leverage requires **FILING, COVENANT and ANNEX**. Their presence makes checks possible; it does not make every claim true.
- Reviewed inputs are debt 75, cash 15, restricted cash 5, EBITDA 20: eligible cash **10**, leverage **3.25×**.
- Remove ANNEX: the supported calculation path is unresolved; the previous numeric variable is not proof a new calculation is supported.
- Restore the actual eligible annex passage: the ratio path becomes checkable again. Comfort still requires downside evidence and an agreed criterion.
- Retrieval precision and recall compare IDs against the supplied relevance labels. Query edits can legitimately change them; inspect the **Missed** column. The small query set is instructional, not a production benchmark.
- Optional dense/live switches stay False by default; no fresh model response should appear unless enabled.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.